In [1]:
library(dplyr)
library(survival)
library(splines)
library(nlme)
library(MASS)
install.packages("joineRML", repos = "https://cloud.r-project.org")
library(joineRML)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘nlme’


The following object is masked from ‘package:dplyr’:

    collapse



Attaching package: ‘MASS’


The following object is masked from ‘package:dplyr’:

    select


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [2]:
base_path <- "/kaggle/input/datasets/jonasmschl/data-lsr-13"
csv_path_train <- file.path(base_path, "data_joinerml_train")
csv_path_test <- file.path(base_path, "data_joinerml_test")

In [3]:
# Set variables
num_traj = 100 # Number of patients
timeout = 1000 # Max visits
num_sim = 100

In [4]:
# Load test and train data for evaluation
files_test <- paste0(
  "DIVAT_sim_missing_test_", num_traj,
  #"DIVAT_sim_", num_traj,
  "_visit_", timeout,
  "_long_3_seed_", 1:num_sim,
  ".csv"
)

files_test_list <- lapply(
  file.path(csv_path_test, files_test),
  read.csv
)

files_train <- paste0(
  "DIVAT_sim_missing_train_", num_traj,
  #"DIVAT_sim_", num_traj,
  "_visit_", timeout,
  "_long_3_seed_", 1:num_sim,
  ".csv"
)

files_train_list <- lapply(
  file.path(csv_path_train, files_train),
  read.csv
)

In [5]:
# Prepare data
prepare_data <- function(df) {
  df_ready <- df %>%
    dplyr::select(id, obstime, time, event, Y1, Y2, Y3, X1, X2, X3) %>%
    dplyr::arrange(id, obstime) %>%
    dplyr::group_by(id) %>%
    dplyr::mutate(
      event = ifelse(event == "True", 1, 0)
    ) %>%
    dplyr::ungroup()

  return(df_ready)
}

test_prepared <- lapply(
  files_test_list,
  prepare_data
)

train_prepared <- lapply(
  files_train_list,
  prepare_data
)

In [6]:
output_dir <- "/kaggle/working/models"
dir.create(
  output_dir,
  recursive = TRUE
)

In [21]:
# Fitting the model via mjoint
for (seed in 86:100){
    cat("\n====================\n")
    cat("Fitting seed:", seed, "\n")
    cat("Dataset:", files_train[seed], "\n")
    cat("====================\n\n")
    set.seed(seed)

    # Set data
    dat <- train_prepared[[seed]]
    dat_Y1 <- dat %>%
      dplyr::filter(!is.na(Y1))
    
    dat_Y2 <- dat %>%
      dplyr::filter(!is.na(Y2))
    
    dat_Y3 <- dat %>%
      dplyr::filter(!is.na(Y3))
    
    valid_ids <- Reduce(
      intersect,
      list(
        unique(dat_Y1$id),
        unique(dat_Y2$id),
        unique(dat_Y3$id)
      )
    )
    
    dat_Y1 <- dat_Y1 %>%
      dplyr::filter(id %in% valid_ids)
    
    dat_Y2 <- dat_Y2 %>%
      dplyr::filter(id %in% valid_ids)
    
    dat_Y3 <- dat_Y3 %>%
      dplyr::filter(id %in% valid_ids)
    
    fit_joint <- mjoint(
        formLongFixed = list(
          #"Y1" = Y1 ~ ns(obstime, df = 3) + Y3_previous + X1 + X2 + X3, # with Spline
          "Y1" = Y1 ~ ns(obstime, df = 3) + X1 + X2 + X3, # without Spline
          "Y2" = Y2 ~ ns(obstime, df = 3) + X1 + X2 + X3,
          "Y3" = Y3 ~ ns(obstime, df = 3) + X1 + X2 + X3
        ),
        formLongRandom = list(
          "Y1" = ~ obstime | id,
          "Y2" = ~ obstime | id,
          "Y3" = ~ 1 | id
        ),
        formSurv = Surv(time, event) ~ X1 + X2 + X3,
        data = list(dat_Y1, dat_Y2, dat_Y3),
        timeVar = "obstime",
       #control = list(
        #  burnin = 300,
         # mcmaxIter = 500,
          #nMC = 300,
          #type = "sobol",
          #nMCmax = 10000
        #),
        verbose = FALSE
      )
    saveRDS(fit_joint, file = file.path(output_dir, paste0("mjoint_fit_seed_", seed,
                                     "_with_num_", num_traj, "_and_timeout_", timeout,"_missing.rds")))
    rm(fit_joint, dat, dat_Y1, dat_Y2, dat_Y3)
    gc()
}


Fitting seed: 86 
Dataset: DIVAT_sim_missing_train_100_visit_1000_long_3_seed_86.csv 



Running multivariate LMM EM algorithm to establish initial parameters...

Finished multivariate LMM EM algorithm...

Data are unbalanced... using sub-optimal initial parameters for gamma

EM algorithm has converged!


Calculating post model fit statistics...





Fitting seed: 87 
Dataset: DIVAT_sim_missing_train_100_visit_1000_long_3_seed_87.csv 



Running multivariate LMM EM algorithm to establish initial parameters...

Finished multivariate LMM EM algorithm...

Data are unbalanced... using sub-optimal initial parameters for gamma

EM algorithm has converged!


Calculating post model fit statistics...





Fitting seed: 88 
Dataset: DIVAT_sim_missing_train_100_visit_1000_long_3_seed_88.csv 



Running multivariate LMM EM algorithm to establish initial parameters...

Finished multivariate LMM EM algorithm...

Data are unbalanced... using sub-optimal initial parameters for gamma

EM algorithm has converged!


Calculating post model fit statistics...




... (20 outputs left out)


Fitting seed: 99 
Dataset: DIVAT_sim_missing_train_100_visit_1000_long_3_seed_99.csv 



Running multivariate LMM EM algorithm to establish initial parameters...

Finished multivariate LMM EM algorithm...

Data are unbalanced... using sub-optimal initial parameters for gamma

EM algorithm has converged!


Calculating post model fit statistics...





Fitting seed: 100 
Dataset: DIVAT_sim_missing_train_100_visit_1000_long_3_seed_100.csv 



Running multivariate LMM EM algorithm to establish initial parameters...

Finished multivariate LMM EM algorithm...

Data are unbalanced... using sub-optimal initial parameters for gamma

EM algorithm has converged!


Calculating post model fit statistics...




In [6]:
ids_Y1 <- unique(dat_Y1$id)
ids_Y2 <- unique(dat_Y2$id)
ids_Y3 <- unique(dat_Y3$id)

length(ids_Y1)
length(ids_Y2)
length(ids_Y3)

setdiff(ids_Y2, ids_Y1)
setdiff(ids_Y3, ids_Y1)
setdiff(ids_Y1, ids_Y2)

ERROR: Error: object 'dat_Y1' not found


In [22]:
zip(
  zipfile = "/kaggle/working/models.zip",
  files = "/kaggle/working/models"
)